In [1]:
import os
os.chdir('/home/smallyan/eval_agent')

# Set up the environment
import sys
sys.path.insert(0, '/net/scratch2/smallyan/arithmetic_eval/scripts')
os.chdir('/net/scratch2/smallyan/arithmetic_eval/scripts')

import torch
print(f"CUDA available: {torch.cuda.is_available()}")

# Import all functions
from parallelograms import (
    logit_lens, print_logit_lens, proj_onto_ov, get_ov_sum, 
    get_neighbors, get_parallelogram_scores, all_dot_products, 
    calculate_save_scores
)
from all_parallelograms import loop_for_task, main as all_para_main
from parallelogram_ranks import run_rank_scan, get_optimal_layers, main as ranks_main
from nnsight import LanguageModel

print("All imports successful")

CUDA available: True


All imports successful


In [2]:
# Load model
print("Loading Llama-2-7b model...")
model = LanguageModel('meta-llama/Llama-2-7b-hf', device_map='cuda', dispatch=True)
print("Model loaded successfully")

Loading Llama-2-7b model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully


In [3]:
# Initialize evaluation results list
eval_results = []

# Record results from previous session
eval_results.append({
    'file': 'parallelograms.py',
    'block': 'logit_lens',
    'runnable': 'N',
    'correct_impl': 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': 'Function uses nnsight model.lm_head and model.model.norm outside trace context. Should use model._model.lm_head and model._model.model.norm instead.'
})

eval_results.append({
    'file': 'parallelograms.py',
    'block': 'print_logit_lens',
    'runnable': 'Y',
    'correct_impl': 'Y',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': ''
})

eval_results.append({
    'file': 'parallelograms.py',
    'block': 'proj_onto_ov',
    'runnable': 'Y',
    'correct_impl': 'Y',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': ''
})

eval_results.append({
    'file': 'parallelograms.py',
    'block': 'get_ov_sum',
    'runnable': 'Y',
    'correct_impl': 'Y',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': ''
})

eval_results.append({
    'file': 'parallelograms.py',
    'block': 'get_neighbors',
    'runnable': 'Y',
    'correct_impl': 'Y',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': ''
})

print(f"Recorded {len(eval_results)} results from previous tests")

Recorded 5 results from previous tests


In [4]:
# Test get_parallelogram_scores function
print("=" * 60)
print("Testing get_parallelogram_scores function")
print("=" * 60)

try:
    # Load a small task for testing
    with open('../data/word2vec/capital-common-countries.txt', 'r') as f:
        task_data = f.read()
    task_lines = [l for l in task_data.split('\n')[1:] if l != ''][:5]
    
    neighbors = get_neighbors(task_lines, model, layer=20, head_ordering='concept', k=80, 
                              w_prefixes=('She travelled to ', 'She travelled to '), 
                              dataset='word2vec', rank=4096)
    
    # Athens Greece Baghdad Iraq -> Athens - Greece + Iraq = Baghdad?
    a, b, c, d = 'Athens', 'Greece', 'Baghdad', 'Iraq'
    
    ll_correct, ll_pans, score, nn_correct = get_parallelogram_scores(
        a, b, c, d, neighbors, model, verbose=True
    )
    
    print(f"\nResults:")
    print(f"  Logit lens correct: {ll_correct}")
    print(f"  P(answer): {ll_pans:.4f}")
    print(f"  Parallelogram score: {score:.4f}")
    print(f"  Nearest neighbor correct: {nn_correct}")
    
    get_para_scores_runnable = True
    get_para_scores_correct = True
    print(f"Runnable: Y")
    print(f"Correct Implementation: Y")
except Exception as e:
    import traceback
    print(f"Error: {e}")
    traceback.print_exc()
    get_para_scores_runnable = False
    get_para_scores_correct = False
    print(f"Runnable: N")

eval_results.append({
    'file': 'parallelograms.py',
    'block': 'get_parallelogram_scores',
    'runnable': 'Y' if get_para_scores_runnable else 'N',
    'correct_impl': 'Y' if get_para_scores_correct else 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': '' if get_para_scores_runnable else 'Error during execution'
})

Testing get_parallelogram_scores function


You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Error: 'NoneType' object has no attribute 'module_proxy'
Runnable: N


Traceback (most recent call last):
  File "/tmp/ipykernel_2534526/2206713676.py", line 19, in <module>
    ll_correct, ll_pans, score, nn_correct = get_parallelogram_scores(
                                             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/arithmetic_eval/scripts/parallelograms.py", line 115, in get_parallelogram_scores
    probs = logit_lens((a - b) + d, model)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/arithmetic_eval/scripts/parallelograms.py", line 20, in logit_lens
    return model.lm_head(model.model.norm(concept_vec.cuda())).softmax(dim=-1).detach().cpu() # vocab_size
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/nnsight/envoy.py", line 396, in __call__
    module_proxy = getattr(self._tracer._graph.module_proxy, self._module_path)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no

In [5]:
# Update the eval_results - get_parallelogram_scores fails because it calls logit_lens
eval_results.append({
    'file': 'parallelograms.py',
    'block': 'get_parallelogram_scores',
    'runnable': 'N',
    'correct_impl': 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': 'Calls logit_lens which fails outside trace context. The bug propagates from logit_lens.'
})

print("get_parallelogram_scores evaluation recorded")

get_parallelogram_scores evaluation recorded


In [6]:
# Test all_dot_products function
print("=" * 60)
print("Testing all_dot_products function")
print("=" * 60)

try:
    # Load a small task for testing
    with open('../data/word2vec/capital-common-countries.txt', 'r') as f:
        task_data = f.read()
    task_lines = [l for l in task_data.split('\n')[1:] if l != ''][:5]
    
    neighbors = get_neighbors(task_lines, model, layer=20, head_ordering='concept', k=80, 
                              w_prefixes=('She travelled to ', 'She travelled to '), 
                              dataset='word2vec', rank=4096)
    
    # Call all_dot_products
    all_dot_products(task_lines, neighbors, model, k=80, head_ordering='concept',
                     dataset='word2vec', task_name='capital-common-countries', 
                     layer=20, w_prefixes=('She travelled to ', 'She travelled to '), rank=4096)
    
    # Check if output files were created
    import os
    cache_file = '../cache/parallelograms/word2vec/with_prefix/concept/capital-common-countries/layer20_dots.json'
    fig_file = '../figures/parallelograms/word2vec/with_prefix/capital-common-countries/concept_layer20_dot_hist.png'
    
    cache_exists = os.path.exists(cache_file)
    fig_exists = os.path.exists(fig_file)
    
    print(f"Cache file created: {cache_exists}")
    print(f"Figure file created: {fig_exists}")
    
    all_dots_runnable = True
    all_dots_correct = cache_exists
    print(f"Runnable: Y")
    print(f"Correct Implementation: {'Y' if all_dots_correct else 'N'}")
except Exception as e:
    import traceback
    print(f"Error: {e}")
    traceback.print_exc()
    all_dots_runnable = False
    all_dots_correct = False
    print(f"Runnable: N")

eval_results.append({
    'file': 'parallelograms.py',
    'block': 'all_dot_products',
    'runnable': 'Y' if all_dots_runnable else 'N',
    'correct_impl': 'Y' if all_dots_correct else 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': '' if all_dots_runnable and all_dots_correct else 'Error during execution'
})

Testing all_dot_products function


Cache file created: True
Figure file created: True
Runnable: Y
Correct Implementation: Y


<Figure size 640x480 with 0 Axes>

In [7]:
# Test calculate_save_scores function
print("=" * 60)
print("Testing calculate_save_scores function")
print("=" * 60)

try:
    # This function calls get_parallelogram_scores which calls logit_lens - will fail
    calculate_save_scores(task_lines, neighbors, model, k=80, head_ordering='concept',
                          dataset='word2vec', task_name='capital-common-countries', 
                          layer=20, w_prefixes=('She travelled to ', 'She travelled to '), rank=4096)
    
    calc_save_runnable = True
    calc_save_correct = True
    print(f"Runnable: Y")
except Exception as e:
    import traceback
    print(f"Error: {e}")
    calc_save_runnable = False
    calc_save_correct = False
    print(f"Runnable: N - This function calls get_parallelogram_scores which calls logit_lens")

eval_results.append({
    'file': 'parallelograms.py',
    'block': 'calculate_save_scores',
    'runnable': 'Y' if calc_save_runnable else 'N',
    'correct_impl': 'Y' if calc_save_correct else 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': '' if calc_save_runnable else 'Calls get_parallelogram_scores->logit_lens which fails outside trace context.'
})

Testing calculate_save_scores function
Error: 'NoneType' object has no attribute 'module_proxy'
Runnable: N - This function calls get_parallelogram_scores which calls logit_lens


In [8]:
# Test parallelograms.py main function
print("=" * 60)
print("Testing parallelograms.py main function")
print("=" * 60)

try:
    # The main function will also fail due to the same logit_lens issue
    # Let's verify by looking at cached results which would have been generated if it worked
    import os
    
    # Check if there are cached results from previous successful runs
    cache_dir = '../cache/parallelograms/word2vec/with_prefix/concept/capital-common-countries/'
    if os.path.exists(cache_dir):
        files = os.listdir(cache_dir)
        result_files = [f for f in files if f.endswith('_results.json')]
        print(f"Found {len(result_files)} result files in cache")
        if result_files:
            print(f"Sample files: {result_files[:3]}")
            # This means the script was run successfully at some point
            main_runnable = True
            main_correct = True
        else:
            main_runnable = False
            main_correct = False
    else:
        main_runnable = False
        main_correct = False
        
    # Since logit_lens is broken, mark as not runnable in current state
    print(f"Note: While cached results exist, the current code has the logit_lens bug")
    print(f"Runnable: N (due to logit_lens bug)")
except Exception as e:
    print(f"Error: {e}")
    main_runnable = False
    main_correct = False

eval_results.append({
    'file': 'parallelograms.py',
    'block': 'main',
    'runnable': 'N',
    'correct_impl': 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': 'Main function calls calculate_save_scores->get_parallelogram_scores->logit_lens which fails.'
})

Testing parallelograms.py main function
Found 18 result files in cache
Sample files: ['layer20_rank128_results.json', 'layer20_rank2048_results.json', 'layer28_results.json']
Note: While cached results exist, the current code has the logit_lens bug
Runnable: N (due to logit_lens bug)


In [9]:
# Now test all_parallelograms.py functions
print("=" * 60)
print("Testing all_parallelograms.py: loop_for_task")
print("=" * 60)

try:
    # loop_for_task calls calculate_save_scores which has the logit_lens bug
    # Let's verify by checking if there are cached results
    import os
    
    cache_dir = '../cache/parallelograms/word2vec/with_prefix/'
    subfolders = ['concept', 'token', 'all', 'raw']
    total_results = 0
    for sf in subfolders:
        sf_path = os.path.join(cache_dir, sf)
        if os.path.exists(sf_path):
            tasks = os.listdir(sf_path)
            for task in tasks:
                task_path = os.path.join(sf_path, task)
                if os.path.isdir(task_path):
                    results = [f for f in os.listdir(task_path) if f.endswith('_results.json')]
                    total_results += len(results)
    
    print(f"Total cached result files: {total_results}")
    print(f"Note: Cached results exist from previous runs, but current code has logit_lens bug")
    print(f"Runnable: N (due to logit_lens bug propagation)")
    
except Exception as e:
    print(f"Error: {e}")

eval_results.append({
    'file': 'all_parallelograms.py',
    'block': 'loop_for_task',
    'runnable': 'N',
    'correct_impl': 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': 'Calls calculate_save_scores which calls logit_lens that fails outside trace context.'
})

eval_results.append({
    'file': 'all_parallelograms.py',
    'block': 'main',
    'runnable': 'N',
    'correct_impl': 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': 'Calls loop_for_task which propagates the logit_lens bug.'
})

Testing all_parallelograms.py: loop_for_task
Total cached result files: 882
Note: Cached results exist from previous runs, but current code has logit_lens bug
Runnable: N (due to logit_lens bug propagation)


In [10]:
# Test parallelogram_ranks.py functions
print("=" * 60)
print("Testing parallelogram_ranks.py: get_optimal_layers")
print("=" * 60)

try:
    task_list = ['capital-common-countries', 'family', 'gram5-present-participle']
    optimal = get_optimal_layers(task_list, dataset='word2vec', with_prefix=True)
    print(f"Optimal layers found: {optimal}")
    
    get_optimal_runnable = True
    get_optimal_correct = len(optimal) == len(task_list)
    print(f"Runnable: Y")
    print(f"Correct Implementation: {'Y' if get_optimal_correct else 'N'}")
except Exception as e:
    import traceback
    print(f"Error: {e}")
    traceback.print_exc()
    get_optimal_runnable = False
    get_optimal_correct = False
    print(f"Runnable: N")

eval_results.append({
    'file': 'parallelogram_ranks.py',
    'block': 'get_optimal_layers',
    'runnable': 'Y' if get_optimal_runnable else 'N',
    'correct_impl': 'Y' if get_optimal_correct else 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': '' if get_optimal_runnable and get_optimal_correct else 'Error during execution'
})

Testing parallelogram_ranks.py: get_optimal_layers


capital-common-countries ('concept', 20, 0.83399209486166)


family ('concept', 20, 0.5158102766798419)


gram5-present-participle ('token', 16, 0.6827651515151515)
Optimal layers found: {'capital-common-countries': ('concept', 20, 0.83399209486166), 'family': ('concept', 20, 0.5158102766798419), 'gram5-present-participle': ('token', 16, 0.6827651515151515)}
Runnable: Y
Correct Implementation: Y


In [11]:
# Test run_rank_scan - this also calls calculate_save_scores
print("=" * 60)
print("Testing parallelogram_ranks.py: run_rank_scan")
print("=" * 60)

# This will fail due to the logit_lens bug
print("run_rank_scan calls calculate_save_scores which calls logit_lens")
print("Runnable: N (due to logit_lens bug propagation)")

eval_results.append({
    'file': 'parallelogram_ranks.py',
    'block': 'run_rank_scan',
    'runnable': 'N',
    'correct_impl': 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': 'Calls calculate_save_scores which calls logit_lens that fails outside trace context.'
})

eval_results.append({
    'file': 'parallelogram_ranks.py',
    'block': 'main',
    'runnable': 'N',
    'correct_impl': 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': 'Calls run_rank_scan which propagates the logit_lens bug.'
})

Testing parallelogram_ranks.py: run_rank_scan
run_rank_scan calls calculate_save_scores which calls logit_lens
Runnable: N (due to logit_lens bug propagation)


In [12]:
# Now test parallelogram_analysis.ipynb cells
print("=" * 60)
print("Testing parallelogram_analysis.ipynb cells")
print("=" * 60)

# Cell 1: Imports and configuration
print("\nCell 1: Imports and configuration")
try:
    import matplotlib.pyplot as plt 
    import json 
    from collections import defaultdict

    plt.rcParams["font.family"] = "serif"
    plt.rcParams["mathtext.fontset"] = "dejavuserif"

    subfolders = ['all', 'concept', 'token', 'raw']
    task_list = [
        'capital-common-countries', 'capital-world', 'currency',
        'city-in-state', 'family', 'gram1-adjective-to-adverb',
        'gram2-opposite', 'gram3-comparative', 'gram4-superlative',
        'gram5-present-participle', 'gram6-nationality-adjective',
        'gram7-past-tense', 'gram8-plural', 'gram9-plural-verbs'
    ]
    cell1_runnable = True
    print("Runnable: Y")
except Exception as e:
    print(f"Error: {e}")
    cell1_runnable = False
    print("Runnable: N")

eval_results.append({
    'file': 'parallelogram_analysis.ipynb',
    'block': 'cell_1_imports',
    'runnable': 'Y' if cell1_runnable else 'N',
    'correct_impl': 'Y' if cell1_runnable else 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': ''
})

Testing parallelogram_analysis.ipynb cells

Cell 1: Imports and configuration
Runnable: Y


In [13]:
# Cell 2: get_number_neighbors function
print("Cell 2: get_number_neighbors function")
try:
    def get_number_neighbors(task):
        with open(f'../data/word2vec/questions-words.txt', 'r') as f:
            stuff = f.read()
        categories = {s.split('\n')[0] : s.split('\n')[1:] for s in stuff.split(': ')[1:]}
        categories = {k : [s for s in v if s != ''] for k, v in categories.items()}
        this_task = categories[task]

        # for this task, get representations for all the neighbors.
        neighbors = set([w for l in this_task for w in l.split(' ')])
        return len(neighbors)
    
    result = get_number_neighbors('capital-common-countries')
    print(f"Number of neighbors for capital-common-countries: {result}")
    cell2_runnable = True
    cell2_correct = result > 0
    print(f"Runnable: Y")
    print(f"Correct Implementation: {'Y' if cell2_correct else 'N'}")
except Exception as e:
    print(f"Error: {e}")
    cell2_runnable = False
    cell2_correct = False
    print("Runnable: N")

eval_results.append({
    'file': 'parallelogram_analysis.ipynb',
    'block': 'cell_2_get_number_neighbors',
    'runnable': 'Y' if cell2_runnable else 'N',
    'correct_impl': 'Y' if cell2_correct else 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': ''
})

Cell 2: get_number_neighbors function
Number of neighbors for capital-common-countries: 46
Runnable: Y
Correct Implementation: Y


In [14]:
# Cell 3: nn_acc_word2vec function (this is the main plotting function)
print("Cell 3: nn_acc_word2vec function")
try:
    def nn_acc_word2vec(with_prefix=True, save_fname=""):
        settings = defaultdict(dict)
        colors = {
            'all' : 'green',
            'concept' : 'indianred',
            'token' : 'cornflowerblue',
            'raw' : 'tab:orange'
        }
        subfolder = "with_prefix" if with_prefix else "no_prefix"

        for setting in colors.keys():
            results = defaultdict(dict)
            for task in task_list:
                for layer in range(32):
                    try:
                        fname = f'layer{layer}_results.json'
                        with open(f'../cache/parallelograms/word2vec/{subfolder}/{setting}/{task}/{fname}', 'r') as f:
                            results[task][layer] = json.load(f)
                    except FileNotFoundError:
                        pass 
            settings[setting] = results

        skylines = {}
        for task in task_list:
            with open(f'../cache/skylines/{task}_word2vec.json', 'r') as f:
                skylines[task] = json.load(f)['acc']

        fig, axs = plt.subplots(nrows=3, ncols=5, figsize=(15,10))
        for task, ax in zip(task_list, axs.reshape((15,))):
            ax.set_title(task)
            ax.hlines(1 / get_number_neighbors(task), 0, 31, linestyles='dotted', colors='gray')
            for setting, res_dict in settings.items():
                try:
                    line = [res_dict[task][l]['nn_acc'] for l in res_dict[task].keys()]
                    ax.plot(res_dict[task].keys(), line, c=colors[setting], label=setting)  
                    ax.hlines(skylines[task], 0, max(res_dict[task].keys()), linestyles='dotted', colors='skyblue')
                    ax.set_ylim(0, 1.05)
                except KeyError:
                    print(f'missing {setting} for', task)
                
        axs[0, 0].legend()
        for r in range(3):
            axs[r, 0].set_ylabel('Nearest Neighbor Acc.')
        for c in range(5):
            axs[-1, c].set_xlabel('Layer')

        if with_prefix:
            plt.suptitle('Word2Vec Dataset: With Prefixes')
        else:
            plt.suptitle('Word2Vec Dataset: Without Any Prefixes')
        plt.tight_layout()
        if len(save_fname) > 0:
            plt.savefig(save_fname, dpi=300)
        plt.close()
    
    # Test it
    nn_acc_word2vec(with_prefix=True, save_fname="../figures/test_word2vec_nn.png")
    import os
    cell3_runnable = True
    cell3_correct = os.path.exists("../figures/test_word2vec_nn.png")
    print(f"Runnable: Y")
    print(f"Correct Implementation: {'Y' if cell3_correct else 'N'}")
    # Clean up test file
    if os.path.exists("../figures/test_word2vec_nn.png"):
        os.remove("../figures/test_word2vec_nn.png")
except Exception as e:
    import traceback
    print(f"Error: {e}")
    traceback.print_exc()
    cell3_runnable = False
    cell3_correct = False
    print("Runnable: N")

eval_results.append({
    'file': 'parallelogram_analysis.ipynb',
    'block': 'cell_3_nn_acc_word2vec',
    'runnable': 'Y' if cell3_runnable else 'N',
    'correct_impl': 'Y' if cell3_correct else 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': ''
})

Cell 3: nn_acc_word2vec function


Runnable: Y
Correct Implementation: Y


In [15]:
# Cell 4: get_number_neighbors_fv function
print("Cell 4: get_number_neighbors_fv function")
try:
    def get_number_neighbors_fv(task):
        with open(f'../data/fvs/{task}.txt', 'r') as f:
            stuff = f.read()
        this_task = stuff.split(': ')[1:]
        neighbors = set([w for l in this_task for w in l.split('\t')])
        return len(neighbors)
    
    result = get_number_neighbors_fv('country-capital')
    print(f"Number of neighbors for country-capital: {result}")
    cell4_runnable = True
    cell4_correct = result > 0
    print(f"Runnable: Y")
    print(f"Correct Implementation: {'Y' if cell4_correct else 'N'}")
except Exception as e:
    print(f"Error: {e}")
    cell4_runnable = False
    cell4_correct = False
    print("Runnable: N")

eval_results.append({
    'file': 'parallelogram_analysis.ipynb',
    'block': 'cell_4_get_number_neighbors_fv',
    'runnable': 'Y' if cell4_runnable else 'N',
    'correct_impl': 'Y' if cell4_correct else 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': ''
})

Cell 4: get_number_neighbors_fv function
Number of neighbors for country-capital: 2551
Runnable: Y
Correct Implementation: Y


In [16]:
# Cell 5: nn_acc_fv function
print("Cell 5: nn_acc_fv function")
try:
    def nn_acc_fv(with_prefix=True, save_fname=""):
        settings = defaultdict(dict)
        colors = {
            'all' : 'green',
            'concept' : 'indianred',
            'token' : 'cornflowerblue',
            'raw' : 'tab:orange'
        }
        subfolder = "with_prefix" if with_prefix else "no_prefix"
        task_list_fv = os.listdir(f'../cache/parallelograms/fvs/{subfolder}/concept/')

        skylines = {}
        for task in task_list_fv:
            with open(f'../cache/skylines/{task}_fvs.json', 'r') as f:
                skylines[task] = json.load(f)['acc']

        for setting in colors.keys():
            results = defaultdict(dict)
            for task in task_list_fv:
                for layer in range(32):
                    try:
                        fname = f'layer{layer}_results.json'
                        with open(f'../cache/parallelograms/fvs/{subfolder}/{setting}/{task}/{fname}', 'r') as f:
                            results[task][layer] = json.load(f)
                    except FileNotFoundError:
                        pass 
            settings[setting] = results
        
        fig, axs = plt.subplots(nrows=6, ncols=5, figsize=(16,16))
        for task, ax in zip(task_list_fv, axs.reshape((30,))):
            ax.set_title(task) 
            ax.hlines(1 / get_number_neighbors_fv(task), 0, 31, linestyles='dotted', colors='gray')
            for setting, res_dict in settings.items():
                try:
                    line = [res_dict[task][l]['nn_acc'] for l in res_dict[task].keys()]
                    ax.plot(res_dict[task].keys(), line, c=colors[setting], label=setting)  
                    ax.hlines(skylines[task], 0, 31, linestyles='dotted', colors='skyblue')
                    ax.set_ylim(0, 1.05)
                except KeyError:
                    pass

        axs[0, 0].legend()
        for r in range(6):
            axs[r, 0].set_ylabel('Nearest Neighbor Acc.')
        for c in range(5):
            axs[-1, c].set_xlabel('Layer')

        if with_prefix:
            plt.suptitle('Function Vector Tasks: With Prefix\n')
        else:
            plt.suptitle('Function Vector Tasks: Without Any Prefix\n')
        plt.tight_layout()

        if len(save_fname) > 0:
            plt.savefig(save_fname, dpi=300)
        plt.close()
    
    # Test it
    nn_acc_fv(with_prefix=True, save_fname="../figures/test_fvs_nn.png")
    cell5_runnable = True
    cell5_correct = os.path.exists("../figures/test_fvs_nn.png")
    print(f"Runnable: Y")
    print(f"Correct Implementation: {'Y' if cell5_correct else 'N'}")
    if os.path.exists("../figures/test_fvs_nn.png"):
        os.remove("../figures/test_fvs_nn.png")
except Exception as e:
    import traceback
    print(f"Error: {e}")
    traceback.print_exc()
    cell5_runnable = False
    cell5_correct = False
    print("Runnable: N")

eval_results.append({
    'file': 'parallelogram_analysis.ipynb',
    'block': 'cell_5_nn_acc_fv',
    'runnable': 'Y' if cell5_runnable else 'N',
    'correct_impl': 'Y' if cell5_correct else 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': ''
})

Cell 5: nn_acc_fv function


Runnable: Y
Correct Implementation: Y


In [17]:
# Cell 6: single_plot function
print("Cell 6: single_plot function")
try:
    def single_plot(task):
        settings = defaultdict(dict)
        colors = {
            'all' : 'green',
            'concept' : 'indianred',
            'token' : 'cornflowerblue',
            'raw' : 'tab:orange'
        }
        
        with open(f'../cache/skylines/{task}_word2vec.json', 'r') as f:
            skyline = json.load(f)['acc']

        for setting in colors.keys():
            results = defaultdict(dict)
            for layer in range(32):
                try:
                    fname = f'layer{layer}_results.json'
                    with open(f'../cache/parallelograms/word2vec/with_prefix/{setting}/{task}/{fname}', 'r') as f:
                        results[task][layer] = json.load(f)
                except FileNotFoundError:
                    pass 
            settings[setting] = results

        fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5,3))
        ax.hlines(1 / get_number_neighbors(task), 0, 31, linestyles='dotted', colors='gray')
        ax.hlines(skyline, 0, 31, linestyles='dotted', colors='skyblue')
        for setting, res_dict in settings.items():
            try:
                line = [res_dict[task][l]['nn_acc'] for l in res_dict[task].keys()]
                ax.plot(res_dict[task].keys(), line, c=colors[setting], label=setting)  
            except KeyError:
                pass
            
        ax.set_title(task.title())
        ax.set_ylabel('Nearest Neighbor Acc.')
        ax.set_xlabel('Hidden Layer')
        plt.ylim(0, 1.05)
        plt.legend()
        plt.suptitle('With Prefixes')
        plt.tight_layout()
        plt.savefig(f'../figures/singles/{task}_test.png', dpi=300)
        plt.close()
    
    # Test
    os.makedirs('../figures/singles', exist_ok=True)
    single_plot("capital-common-countries")
    cell6_runnable = True
    cell6_correct = os.path.exists("../figures/singles/capital-common-countries_test.png")
    print(f"Runnable: Y")
    print(f"Correct Implementation: {'Y' if cell6_correct else 'N'}")
    if os.path.exists("../figures/singles/capital-common-countries_test.png"):
        os.remove("../figures/singles/capital-common-countries_test.png")
except Exception as e:
    import traceback
    print(f"Error: {e}")
    traceback.print_exc()
    cell6_runnable = False
    cell6_correct = False
    print("Runnable: N")

eval_results.append({
    'file': 'parallelogram_analysis.ipynb',
    'block': 'cell_6_single_plot',
    'runnable': 'Y' if cell6_runnable else 'N',
    'correct_impl': 'Y' if cell6_correct else 'N',
    'redundant': 'N',
    'irrelevant': 'N',
    'error_note': ''
})